# inplace-op-unsafe-warning — ex3: detach: new wrapper sharing array, no recipe, so in-place becomes safe

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `inplace-op-unsafe-warning`. Running the final beacon cell reports progress against the `Backprop: In-place op unsafe warning` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: In-place op unsafe warning` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inplace-op-unsafe-warning`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inplace-op-unsafe-warning"
DD_SUBTOPIC = "Backprop: In-place op unsafe warning"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `detach()` — peel the Recipe off so in-place becomes safe

Ex1 made in-place ops refuse when `.recipe is not None`. Ex2 added a context-manager escape hatch. The third facet is the surgical escape: `detach()` returns a NEW MiniTensor sharing the SAME `.array` but with `recipe=None` and `requires_grad=False`. It says 'I know what I'm doing — disconnect this tensor from the graph here'.

```python
def detach(x: MiniTensor) -> MiniTensor:
    # New wrapper, same underlying storage, no recipe.
    return MiniTensor(x.array, requires_grad=False, recipe=None)
```

**Same array — but different wrapper.** `detach()` is NOT a copy of the data. The new MiniTensor's `.array` IS the same `torch.Tensor` object as `x.array`. So mutating the detached one's `.array` ALSO mutates the original's `.array` — but the original's recipe is no longer hooked into the new one's identity, so the in-place guard on the detached wrapper passes.

**Why this is dangerous AND useful.** Dangerous: you've severed the graph at this point. The reverse pass won't propagate through the detached node. Useful: in inference, in stop-gradient operations (BatchNorm running stats, target networks in RL), and at boundaries where you genuinely want autograd to ignore a subgraph. PyTorch's `tensor.detach()` is the exact same idiom.

**Contrast with the context manager from ex2.** `inplace_unsafe()` disables the guard globally. `detach()` disables it for ONE specific tensor by replacing its wrapper. Local vs global. Both legitimate; different ergonomic trade-offs.

### Exercise 3 — detach: new wrapper sharing array, no recipe, so in-place becomes safe

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the detach pattern: return a new MiniTensor that shares the underlying .array (same object) but has recipe=None and requires_grad=False, so the in-place guard from ex1 no longer fires on the detached wrapper.
> Keywords: detach, stop-gradient, shared-storage, graph-disconnect
> ```

**KCs targeted:** `detach-clears-recipe`, `shared-storage-different-wrapper`

Implement two functions:

1. **`ex3_detach(x: MiniTensor) -> MiniTensor`** — return a NEW `MiniTensor` such that:
   - `result.array is x.array` (SAME torch.Tensor object — no copy).
   - `result.recipe is None` (the recipe-chain ends here).
   - `result.requires_grad is False` (no gradient flow through this point).
   - `result is not x` (a fresh MiniTensor wrapper).

2. **`ex3_add_inplace_safe(x: MiniTensor, y: MiniTensor) -> MiniTensor`** — same guard as ex1: if `x.recipe is not None`, raise `RuntimeError` with 'in-place' in the message; otherwise `x.array += y.array` and return `x`.

Why both: the test scenario is the standard 'I have an intermediate, want to mutate it explicitly'. The fix is `detach()` to peel the recipe off, then the in-place guard from ex1 passes.

Constraints:
- `detach()` must NOT clone the underlying tensor. Identity of `.array` is critical for storage-sharing semantics.
- Mutating the detached wrapper's `.array` MUST mutate the original's `.array` too (they share storage).
- The original tensor's `.recipe` and `.requires_grad` MUST be unchanged.

In [ ]:
def ex3_detach(x: MiniTensor) -> MiniTensor:
    """Return a fresh wrapper sharing x.array, with recipe=None, requires_grad=False."""
    raise NotImplementedError()


def ex3_add_inplace_safe(x: MiniTensor, y: MiniTensor) -> MiniTensor:
    """Same in-place guard as ex1: refuse when x.recipe is not None."""
    raise NotImplementedError()


def _test_ex3():
    def _test_ex3():
        # === detach: returns a new wrapper, same .array ===
        raw = t.tensor([1.0, 2.0, 3.0])
        x = MiniTensor(raw, requires_grad=True)
        x.recipe = Recipe(t.add, (raw,), {}, {0: x})

        detached = ex3_detach(x)
        assert isinstance(detached, MiniTensor)
        assert detached is not x, 'detach must return a new wrapper, not the same object'
        assert detached.array is x.array, (
            f'detached.array must BE x.array (same torch.Tensor object), got identity mismatch'
        )
        assert detached.recipe is None, (
            f'detached.recipe must be None, got {detached.recipe!r}'
        )
        assert detached.requires_grad is False, (
            f'detached.requires_grad must be False, got {detached.requires_grad}'
        )

        # === Original's recipe and requires_grad are UNCHANGED ===
        assert x.recipe is not None, 'detach must not mutate x.recipe'
        assert x.requires_grad is True, 'detach must not mutate x.requires_grad'

        # === Storage IS shared: mutating detached.array also changes x.array ===
        detached.array[0] = 99.0
        assert x.array[0].item() == 99.0, (
            'detached and original must share storage — mutation on one visible on other'
        )
        # restore for further tests
        detached.array[0] = 1.0

        # === The use case: in-place add on the original REFUSES (recipe is set) ===
        raised = False
        try:
            ex3_add_inplace_safe(x, MiniTensor(t.tensor([10.0, 10.0, 10.0])))
        except RuntimeError as e:
            raised = True
            msg = str(e).lower()
            assert 'in-place' in msg or 'inplace' in msg or 'in place' in msg, (
                f'error must mention in-place, got: {e}'
            )
        assert raised, 'in-place on x (recipe-carrying) must refuse'

        # === Now via detach: in-place on the detached wrapper SUCCEEDS ===
        detached = ex3_detach(x)
        y = MiniTensor(t.tensor([10.0, 10.0, 10.0]))
        result = ex3_add_inplace_safe(detached, y)
        assert result is detached
        assert t.allclose(detached.array, t.tensor([11.0, 12.0, 13.0])), (
            f'in-place add via detach failed: {detached.array}'
        )
        # And critically, x.array IS detached.array → it ALSO changed (shared storage).
        assert t.allclose(x.array, t.tensor([11.0, 12.0, 13.0])), (
            'x.array also reflects mutation (shared storage with detached)'
        )

        # === Leaf (no recipe) detach is a no-op semantically but still creates a new wrapper ===
        leaf = MiniTensor(t.tensor([1.0, 2.0]), requires_grad=True)
        assert leaf.recipe is None
        d = ex3_detach(leaf)
        assert d is not leaf
        assert d.array is leaf.array
        assert d.recipe is None
        assert d.requires_grad is False
        # Leaf's requires_grad stays True after detach (we never mutated the original).
        assert leaf.requires_grad is True

        # === Double detach is idempotent (still works) ===
        dd = ex3_detach(d)
        assert dd.array is leaf.array  # transitively shared
        assert dd.recipe is None
        assert dd.requires_grad is False

        # === Detach has no requires_grad flag — always False ===
        # Even when input was requires_grad=False, detach still returns rg=False.
        no_rg = MiniTensor(t.tensor([1.0]), requires_grad=False)
        d = ex3_detach(no_rg)
        assert d.requires_grad is False

        # === The error message specifically mentions in-place ===
        bad = MiniTensor(t.tensor([1.0]))
        bad.recipe = Recipe(t.add, (), {}, {})
        raised = False
        try:
            ex3_add_inplace_safe(bad, MiniTensor(t.tensor([2.0])))
        except RuntimeError:
            raised = True
        assert raised
        print('ex3 ✓')

    _test_ex3()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_detach(x):
    # New wrapper, SAME underlying torch.Tensor (no copy).
    # recipe=None severs the graph at this point; requires_grad=False
    # tells downstream wrap_forward_fn 'don't track gradients through me'.
    return MiniTensor(x.array, requires_grad=False, recipe=None)


def ex3_add_inplace_safe(x, y):
    if x.recipe is not None:
        raise RuntimeError(
            'in-place op forbidden on a Tensor with a recipe — '
            'would corrupt cached values on the graph; '
            'use detach() first if you really mean it'
        )
    x.array += y.array
    return x
```

**Storage sharing is the whole point — and the danger.** `detach()` returns a wrapper whose `.array IS x.array` (same Python object). Mutating the detached wrapper's `.array` mutates the original's `.array` too. The guard from ex1 only inspects the WRAPPER'S `.recipe`, not the underlying storage — so by swapping wrappers we bypass the guard while still mutating the same memory. The user has explicitly chosen to take ownership of the consequence.

**When this is correct vs. when it's a bug.** Correct: in `with torch.no_grad():` blocks, in stop-gradient ops (BN running stats, target networks), at clearly-marked subgraph boundaries. Buggy: anywhere the user 'just wanted the .array' and didn't realize detaching also disconnects future ops from the graph.

**Detach vs. clone+detach.** `clone()` copies storage; `detach()` shares storage. `x.detach().clone()` is the common idiom for 'I want both: a detached AND a separate-storage tensor I can mutate freely without affecting x'. The drill keeps detach as the minimal primitive — clone is a separate concern.

**Contrast with ex2's context manager.** `inplace_unsafe()` disables the guard globally for a code block. `detach()` disables it surgically by replacing the wrapper. Same goal, different ergonomic granularity. PyTorch ships both.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()